# 01 상품 정제 검증

정제 코드(`app/clean.py`) 결과를 문서 숫자와 대조하고, 첫 적재 전 검수표를 만든다.

- 근거: 데이터 처리 기획서 3-8 (전수 검수), 6-3 3번 (건수 대조) / 상품 데이터 정제, 필터 기준 3-5, 3-6, 3-9
- 검수표는 `ml/data/csv/검수_*.csv` (git에 안 올림). 이미 있으면 덮어쓰지 않음 (`OVERWRITE = True`일 때만)
- 확인하는 사람은 엑셀에서 `확인(O/X)` 칸에 O(맞음) 또는 X(틀림), 틀리면 `메모`에 이유 → **CSV UTF-8**로 저장
- GitHub에 올리기 전에 **출력 지우기** (원본 행 값이 보이는 출력은 올리지 않음, be_common 폴더 구조)

In [ ]:
import json
import sys
from collections import Counter
from datetime import date
from pathlib import Path

import pandas as pd

# ml 폴더 찾기 (노트북 위치: ml/data/ipynb)
ML_DIR = next(d for d in [Path.cwd(), *Path.cwd().parents] if (d / "app" / "clean.py").exists())
sys.path.insert(0, str(ML_DIR))

from app.clean import CSV_DIR, WORDS, clean_products, load_tables

OVERWRITE = False   # True면 이미 있는 검수표도 다시 만듦 (적어 둔 O/X가 지워짐)
SEED = 42           # 표본 고르기 고정 값 (다시 돌려도 같은 표본)
pd.set_option("display.max_colwidth", 60)

## 0. 정제 실행
수집일 = 서민 CSV를 저장한 날, 판매 종료 기준일 = 오늘

In [ ]:
tables = load_tables()
collected_on = date.fromtimestamp((CSV_DIR / "kinfa_loan_products.csv").stat().st_mtime)
products = clean_products(tables, ("KINFA", "FSS"), collected_on, date.today())
print("수집일:", collected_on, "/ 정제 상품:", len(products), "개")

## 1. 건수 대조 (기획서 6-3 3번)
문서 숫자는 2026-10-20 수집 기준이라 수집일이 다르면 차이가 날 수 있음. 차이가 크면 원인을 확인

In [ ]:
DOC_COUNTS = {"KINFA": 310, "CREDIT": 107, "RENT": 49, "MORTGAGE": 100, "BIZ": 991}

now = Counter(p["loan_type"] for p in products)
count_table = pd.DataFrame({"문서": DOC_COUNTS, "지금": {k: now.get(k, 0) for k in DOC_COUNTS}})
count_table.loc["합계"] = count_table.sum()
count_table["차이"] = count_table["지금"] - count_table["문서"]
count_table

In [ ]:
# 칩별 상품 수 (정제 기준 3-5, 금융권은 3-6 세분류)
DOC_CHIPS = {
    "TARGET": {"제한 없음": 256, "청년/학생": 23, "직장인": 28, "저신용/서민": 46, "사업자": 1206,
               "특별 지원 대상": 38, "기타": 10},
    "PURPOSE": {"생활비": 170, "전월세": 82, "학자금": 30, "주택 구입": 112, "대환": 34, "창업": 19,
                "사업 운영": 1190, "기타": 1},
    "AMOUNT": {"1천만원 미만": 43, "1천만원 이상": 823, "3천만원 이상": 699, "5천만원 이상": 670,
               "1억원 이상": 527, "담보/보증 비율": 379, "한도 제한 없음": 108, "매출/실비 기준": 49,
               "심사 후 결정": 148},
    "REPAY": {"원리금균등": 606, "원금균등": 547, "만기일시": 908, "분할상환(방식 미표기)": 163, "기타": 29},
    "SECTOR": {"시중은행": 183, "지방은행": 81, "특수은행": 247, "인터넷은행": 62, "저축은행": 365,
               "캐피탈": 229, "생명보험": 24, "신협": 19, "카드사": 22, "손해보험": 15, "신용보증재단": 163,
               "서민금융기관": 46, "공공기관": 45, "지자체": 22, "협약 금융회사": 18, "복지기관": 15, "기타": 1},
    "COMPANY": {"은행": 733, "공공기관": 78, "서민금융센터/미소금융": 44, "저축은행": 369,
                "신협/새마을금고": 30, "카드사": 23, "캐피탈": 229, "보험사": 39, "신용보증재단": 65, "지자체": 10},
    "REGION": {"전국": 1366, "서울": 19, "부산": 4, "대구": 11, "인천": 3, "광주": 8, "대전": 4, "울산": 16,
               "세종": 2, "경기": 30, "강원": 22, "충북": 21, "충남": 3, "전북": 26, "전남": 3, "경북": 7,
               "경남": 8, "제주": 3},
    "CREDIT": {"900 초과": 373, "801~900": 356, "701~800": 335, "601~700": 252, "600 이하": 189,
               "신용 무관": 179, "심사 후 결정": 936},
}
DOC_MISSING = {"TARGET": 0, "PURPOSE": 2, "AMOUNT": 7, "REPAY": 166, "SECTOR": 0, "COMPANY": 4,
               "REGION": 2, "CREDIT": 15}

rows = []
for tag_type, doc in DOC_CHIPS.items():
    now = Counter(t["tag_value"] for p in products for t in p["tags"] if t["tag_type"] == tag_type)
    for chip in WORDS[tag_type]["chips"]:
        rows.append([tag_type, chip, doc.get(chip, 0), now.get(chip, 0)])
    missing = sum(not any(t["tag_type"] == tag_type for t in p["tags"]) for p in products)
    rows.append([tag_type, "(결측)", DOC_MISSING[tag_type], missing])

chip_table = pd.DataFrame(rows, columns=["필터", "칩", "문서", "지금"])
chip_table["차이"] = chip_table["지금"] - chip_table["문서"]
chip_table

In [ ]:
# 확인 필요 (문서: 4개 = 최저 > 최고 2, 최고 20% 초과 1, 기관 없음 1)
checks = [[p["source"], p["source_code"], p["company_name"], p["product_name"], p["rate_min"], p["rate_max"],
           ", ".join(p["check_reasons"])] for p in products if p["check_reasons"]]
pd.DataFrame(checks, columns=["출처", "source_code", "기관", "상품명", "최저 금리", "최고 금리", "사유"])

## 2. 검수표 만들기 (기획서 3-8)
1. 고유 문구 전수: 서민 나이, 소득, 신용, 한도, 기간, 상환, 용도, 대상 + 금리 방식 → `검수_문구.csv`
2. 세부 지원 조건 전수: 대상 제한 놓침 0건 확인 → `검수_세부조건.csv`
3. 상품 표본 50개: 서민 30(자금 용도별 고르게) + 금감원 4종 각 5 → `검수_표본50.csv`

In [ ]:
def raw_of(p):
    """원본 행 (서민은 행 그대로, 금감원은 {base, options})"""
    return json.loads(p["raw_json"])


def show(value):
    """빈 값은 빈 글자로"""
    return "" if value is None else str(value)


def pair(low, high):
    """최저 ~ 최고 (둘 다 없으면 빈 글자)"""
    return "" if low is None and high is None else f"{show(low)} ~ {show(high)}"


def chips(p, tag_type):
    """그 필터의 칩들 ('/'로 이음, 결측이면 빈 글자)"""
    return " / ".join(t["tag_value"] for t in p["tags"] if t["tag_type"] == tag_type)


def save(df, name):
    """검수표 CSV 저장 (이미 있으면 OVERWRITE일 때만)"""
    path = CSV_DIR / name
    if path.exists() and not OVERWRITE:
        print(f"{name} 이미 있음, 건너뜀 (다시 만들려면 OVERWRITE = True)")
        return
    df.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"{name} 저장: {len(df)}행")


kinfa = [p for p in products if p["source"] == "KINFA"]
fss = [p for p in products if p["source"] == "FSS"]

In [ ]:
# 2-1. 고유 문구 전수 (같은 원문이 상품명 등에 따라 다른 결과면 줄이 따로 나옴)
KINFA_PHRASES = [   # (검수 칸, 원본 칸, 정제 결과)
    ("나이", "age", lambda p: pair(p["age_min"], p["age_max"])),
    ("소득(연 만원)", "incm", lambda p: f"{pair(p['income_min'], p['income_max'])} {show(p['income_basis'])}".strip()),
    ("신용", "crdtsc", lambda p: f"{pair(p['credit_min'], p['credit_max'])} [{chips(p, 'CREDIT')}]"),
    ("한도(만원)", "lnlmt", lambda p: f"{show(p['limit_amount'])} [{chips(p, 'AMOUNT')}]"),
    ("기간(개월)", "maxtotlntrm", lambda p: show(p["term_max"])),
    ("상환", "rdptmthd", lambda p: chips(p, "REPAY")),
    ("용도", "usge", lambda p: chips(p, "PURPOSE")),
    ("대상", "trgt", lambda p: f"{chips(p, 'TARGET')} {show(p['target_detail'])}".strip()),
    ("금리 방식", "irtCtg", lambda p: show(p["rate_type"])),
]

rows = []
for field, col, result in KINFA_PHRASES:
    groups = Counter((show(raw_of(p).get(col)), result(p)) for p in kinfa)
    for (text, res), n in sorted(groups.items()):
        rows.append([field, col, text, n, res, "", ""])

phrase_table = pd.DataFrame(rows, columns=["칸", "원본 칸", "원문", "상품 수", "정제 결과", "확인(O/X)", "메모"])
save(phrase_table, "검수_문구.csv")
# 문서 고유 문구 수: 나이 25, 소득 57, 신용 66, 한도 67, 기간 49, 상환 49, 용도 30, 대상 72, 금리 방식 17
phrase_table.groupby("칸", sort=False)["원문"].nunique()

In [ ]:
# 2-2. 세부 지원 조건 전수: 대상 제한이 글에만 있는데 지원 대상 칩에 빠졌으면 X (상품명, 칩, target_detail은 같은 순서로 짝)
by_text = {}
for p in kinfa:
    by_text.setdefault(show(raw_of(p).get("suprtgtdtlcond")), []).append(p)

rows = []
for text, group in sorted(by_text.items()):
    rows.append([text, len(group), " | ".join(p["product_name"] for p in group),
                 " | ".join(chips(p, "TARGET") for p in group),
                 " | ".join(show(p["target_detail"]) for p in group), "", ""])

detail_table = pd.DataFrame(rows, columns=["세부 지원 조건", "상품 수", "상품명", "지원 대상 칩", "target_detail",
                                           "확인(O/X)", "메모"])
save(detail_table, "검수_세부조건.csv")
print("세부 지원 조건 고유 글:", len(detail_table), "종 (문서 289종)")

In [ ]:
# 2-3. 상품 표본 50개 고르기
def kinfa_sample(n=30):
    """서민: 섞은 뒤 첫 자금 용도 칩별로 돌아가며 1개씩 (용도별로 고르게)"""
    groups = {}
    for p in pd.Series(kinfa).sample(frac=1, random_state=SEED):
        groups.setdefault(chips(p, "PURPOSE").split(" / ")[0] or "(결측)", []).append(p)
    picked = []
    while len(picked) < n and any(groups.values()):
        for key in groups:
            if groups[key] and len(picked) < n:
                picked.append(groups[key].pop(0))
    return picked


sample = kinfa_sample()
for loan_type in ("CREDIT", "RENT", "MORTGAGE", "BIZ"):
    group = [p for p in fss if p["loan_type"] == loan_type]
    sample += pd.Series(group).sample(n=5, random_state=SEED).tolist()

print("표본:", Counter(p["loan_type"] for p in sample))
print("서민 표본 자금 용도:", Counter(chips(p, "PURPOSE").split(" / ")[0] or "(결측)" for p in sample[:30]))

In [ ]:
# 2-3. 표본 칸별 대조표 (상품 1개 = 칸마다 1줄)
def fss_rate_source(p):
    """금감원 금리 원문 요약 (옵션 행)"""
    raw = raw_of(p)
    options = raw["options"]
    if p["loan_type"] == "CREDIT":
        lend = [o for o in options if o.get("crdt_lend_rate_type") == "A"]
        if not lend:
            return "대출금리(A) 행 없음"
        return " ".join(f"{k}:{show(lend[0].get(f'crdt_grad_{k}'))}" for k in ("1", "4", "5", "6", "10", "11", "12", "13"))
    if p["loan_type"] == "BIZ":
        if not options:
            return "옵션 행 없음"
        o = options[0]
        grades = " ".join(f"{i}:{show(o.get(f'val1_grad_{i}'))}" for i in range(1, 9))
        return f"최저~최고 {show(o.get('lend_rate_min'))}~{show(o.get('lend_rate_max'))} / 구간 {grades}"
    return "; ".join(f"{o.get('rpay_type')}/{o.get('lend_rate_type')} {o.get('lend_rate_min')}~{o.get('lend_rate_max')}"
                     for o in options) or "옵션 행 없음"


def rates_text(p):
    """구간 금리 줄 요약 (추정은 *)"""
    return " ".join(f"{r['rate_kind']}:{r['rate_key']} {pair(r['rate_min'], r['rate_max'])}"
                    + ("*" if r["rate_source"] == "ESTIMATED" else "") for r in p["rates"])


def base_text(p, *cols):
    """금감원 기본 행 칸들을 ' | '로"""
    base = raw_of(p)["base"]
    return " | ".join(show(base.get(c)) for c in cols if show(base.get(c)))


KINFA_FIELDS = [   # (칸, 원문, 정제 결과)
    ("회사명", lambda p: raw_of(p).get("ofrinstnm"), lambda p: p["company_name"]),
    ("금리", lambda p: raw_of(p).get("irt"), lambda p: f"{pair(p['rate_min'], p['rate_max'])} {show(p['note'])}".strip()),
    ("금리 방식", lambda p: raw_of(p).get("irtCtg"), lambda p: p["rate_type"]),
    ("한도(만원)", lambda p: raw_of(p).get("lnlmt"), lambda p: p["limit_amount"]),
    ("기간(개월)", lambda p: raw_of(p).get("maxtotlntrm"), lambda p: p["term_max"]),
    ("신용 점수", lambda p: raw_of(p).get("crdtsc"), lambda p: pair(p["credit_min"], p["credit_max"])),
    ("나이", lambda p: raw_of(p).get("age"), lambda p: pair(p["age_min"], p["age_max"])),
    ("소득(연 만원)", lambda p: raw_of(p).get("incm"),
     lambda p: f"{pair(p['income_min'], p['income_max'])} {show(p['income_basis'])}".strip()),
    ("중도상환수수료", lambda p: raw_of(p).get("rpymdcfe"),
     lambda p: f"{show(p['prepay_fee_max'])} {show(p['no_prepay_fee_yn'])}".strip()),
    ("비대면", lambda p: raw_of(p).get("jnmthd"), lambda p: p["online_yn"]),
    ("세대 조건", lambda p: raw_of(p).get("housholdcnt"), lambda p: p["extra_cond"]),
    ("공시 종료일", lambda p: raw_of(p).get("prdoprprid"), lambda p: p["disclose_end"]),
    ("지원 대상", lambda p: raw_of(p).get("trgt"), lambda p: f"{chips(p, 'TARGET')} {show(p['target_detail'])}".strip()),
    ("자금 용도", lambda p: raw_of(p).get("usge"), lambda p: chips(p, "PURPOSE")),
    ("필요 금액", lambda p: raw_of(p).get("lnlmt"), lambda p: chips(p, "AMOUNT")),
    ("상환 방법", lambda p: raw_of(p).get("rdptmthd"), lambda p: chips(p, "REPAY")),
    ("금융권", lambda p: raw_of(p).get("ofrinstnm"), lambda p: chips(p, "SECTOR")),
    ("취급기관", lambda p: raw_of(p).get("hdlinst"), lambda p: chips(p, "COMPANY")),
    ("지원 지역", lambda p: raw_of(p).get("rsdAreaPamtEqltIstm"), lambda p: chips(p, "REGION")),
    ("신용점수 구간", lambda p: raw_of(p).get("crdtsc"), lambda p: chips(p, "CREDIT")),
]

FSS_FIELDS = [
    ("회사명", lambda p: base_text(p, "kor_co_nm"), lambda p: p["company_name"]),
    ("금리", fss_rate_source, lambda p: pair(p["rate_min"], p["rate_max"])),
    ("구간 금리", fss_rate_source, rates_text),
    ("금리 방식", lambda p: base_text(p, "lend_rate_type") or fss_rate_source(p), lambda p: p["rate_type"]),
    ("한도(만원)", lambda p: base_text(p, "loan_lmt", "loan_limit", "loan_limit_detl"), lambda p: p["limit_amount"]),
    ("기간(개월)", lambda p: base_text(p, "loan_term"), lambda p: p["term_max"]),
    ("신용 점수", fss_rate_source, lambda p: pair(p["credit_min"], p["credit_max"])),
    ("중도상환수수료", lambda p: base_text(p, "erly_rpay_fee"),
     lambda p: f"{show(p['prepay_fee_max'])} {show(p['no_prepay_fee_yn'])}".strip()),
    ("비대면", lambda p: base_text(p, "join_way"), lambda p: p["online_yn"]),
    ("정책 상품", lambda p: base_text(p, "fin_prdt_type_nm"), lambda p: p["policy_yn"]),
    ("공시 시작~종료", lambda p: base_text(p, "dcls_strt_day", "dcls_end_day"),
     lambda p: pair(p["disclose_start"], p["disclose_end"])),
    ("지원 대상", lambda p: f"{p['loan_type']} {base_text(p, 'join_deny')}".strip(), lambda p: chips(p, "TARGET")),
    ("자금 용도", lambda p: f"{p['loan_type']} {base_text(p, 'use_way')}".strip(), lambda p: chips(p, "PURPOSE")),
    ("필요 금액", lambda p: base_text(p, "loan_type", "loan_lmt", "loan_limit", "loan_limit_detl"),
     lambda p: chips(p, "AMOUNT")),
    ("상환 방법", lambda p: base_text(p, "rpay_type") or fss_rate_source(p), lambda p: chips(p, "REPAY")),
    ("금융권", lambda p: base_text(p, "top_fin_grp_no", "kor_co_nm"), lambda p: chips(p, "SECTOR")),
    ("취급기관", lambda p: base_text(p, "top_fin_grp_no", "kor_co_nm"), lambda p: chips(p, "COMPANY")),
    ("신용점수 구간", fss_rate_source, lambda p: chips(p, "CREDIT")),
]

rows = []
for no, p in enumerate(sample, start=1):
    fields = KINFA_FIELDS if p["source"] == "KINFA" else FSS_FIELDS
    for field, source, result in fields:
        rows.append([no, p["source"], p["loan_type"], p["source_code"], p["company_name"], p["product_name"],
                     field, show(source(p)), show(result(p)), "", ""])

sample_table = pd.DataFrame(rows, columns=["번호", "출처", "상품 종류", "source_code", "기관", "상품명", "칸",
                                           "원문", "정제 결과", "확인(O/X)", "메모"])
save(sample_table, "검수_표본50.csv")

## 3. 오류율 (확인하는 사람이 O/X를 적은 뒤 실행)
기준 (기획서 3-8): 고유 문구 오류 1건 이상 → 단어표 고친 뒤 재검수 / 표본 칸별 오류율 5% 초과 → 그 칸 규칙 고치고 재검수 / 한도 오류 1건 이상 → 재검수

In [ ]:
def error_rates(name):
    """검수표의 칸별 'X 수 / 확인한 수' (확인 안 한 줄 수도 같이)"""
    path = CSV_DIR / name
    if not path.exists():
        print(name, "없음")
        return None
    df = pd.read_csv(path, dtype=str, encoding="utf-8-sig").fillna("")
    mark = df["확인(O/X)"].str.strip().str.upper()
    key = "칸" if "칸" in df.columns else None
    groups = df.groupby(key, sort=False) if key else [("세부 지원 조건", df)]
    rows = []
    for field, g in groups:
        m = mark[g.index]
        rows.append([field, (m == "X").sum(), m.isin(["O", "X"]).sum(), (~m.isin(["O", "X"])).sum()])
    result = pd.DataFrame(rows, columns=["칸", "X", "확인", "안 함"])
    result["한 줄"] = result["칸"] + " " + result["X"].astype(str) + "/" + result["확인"].astype(str)
    return result


for name in ("검수_문구.csv", "검수_세부조건.csv", "검수_표본50.csv"):
    print(f"\n[{name}]")
    print(error_rates(name))